In [1]:
import os

os.environ["LOGURU_LEVEL"] = "ERROR"


# Curating the FLEURS Dataset with NeMo Curator

This notebook walks through the FLEURS audio curation pipeline step by step:
1. Download a small FLEURS split
2. Run ASR inference
3. Compute WER and duration
4. Filter by WER threshold
5. Inspect and visualize results

**Requirements**: GPU recommended for ASR inference. Install with `uv sync --extra audio_cuda12`.

In [2]:
import json
import os
import shutil

from nemo_curator.backends.base import WorkerMetadata
from nemo_curator.backends.xenna import XennaExecutor
from nemo_curator.core.client import RayClient
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.audio.common import GetAudioDurationStage, PreserveByValueStage
from nemo_curator.stages.audio.datasets.fleurs.create_initial_manifest import CreateInitialManifestFleursStage
from nemo_curator.stages.audio.inference.asr.asr_nemo import InferenceAsrNemoStage
from nemo_curator.stages.audio.io.convert import AudioToDocumentStage
from nemo_curator.stages.audio.metrics.wer import GetPairwiseWerStage
from nemo_curator.stages.resources import Resources
from nemo_curator.stages.text.io.writer import JsonlWriter

[NeMo W 2026-06-10 17:44:00 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.


2026-06-10 17:44:00,775 - WARNING - OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.


2026-06-10 17:44:00,791 - INFO - Final configuration contains 0 exporter(s)


2026-06-10 17:44:00,792 - WARNING - No exporters were provided. This means that no telemetry data will be collected.


## Configuration

Adjust these parameters for your setup:

In [3]:
RAW_DATA_DIR = os.path.abspath("./example_audio/fleurs")
LANG = "hy_am"
SPLIT = "dev"  # matches audio_fleurs_benchmark.py default (nightly CI uses train)
MODEL_NAME = "nvidia/stt_hy_fastconformer_hybrid_large_pc"
WER_THRESHOLD = 5.5
GPUS = 1.0

RESULT_DIR = os.path.join(RAW_DATA_DIR, "result")
if os.path.isdir(RESULT_DIR):
    shutil.rmtree(RESULT_DIR)

## Step 1: Build the pipeline

The pipeline has 7 stages: download → ASR → WER → duration → filter → convert → write.
We define a function so we can rebuild the pipeline for each backend run.

In [4]:
class FleursAsrStage(InferenceAsrNemoStage):
    """ASR stage that disables RNNT CUDA graphs when unsupported on local GPUs."""

    def setup(self, worker_metadata: WorkerMetadata | None = None) -> None:
        super().setup(worker_metadata)
        try:
            computer = self.asr_model.decoding.decoding.decoding_computer
            if getattr(computer, "cuda_graphs_mode", None) is not None:
                computer.cuda_graphs_mode = None
        except AttributeError:
            pass


def build_pipeline(result_dir: str) -> Pipeline:
    """Create a fresh pipeline writing to *result_dir*."""
    if os.path.isdir(result_dir):
        shutil.rmtree(result_dir)
    p = Pipeline(name="fleurs_tutorial", description="Download FLEURS, run ASR, filter by WER")
    p.add_stage(
        CreateInitialManifestFleursStage(lang=LANG, split=SPLIT, raw_data_dir=RAW_DATA_DIR).with_(batch_size=4)
    )
    p.add_stage(FleursAsrStage(model_name=MODEL_NAME).with_(resources=Resources(gpus=GPUS)))
    p.add_stage(GetPairwiseWerStage(text_key="text", pred_text_key="pred_text", wer_key="wer_pct"))
    p.add_stage(GetAudioDurationStage(audio_filepath_key="audio_filepath", duration_key="duration"))
    p.add_stage(PreserveByValueStage(input_value_key="wer_pct", target_value=WER_THRESHOLD, operator="le"))
    p.add_stage(AudioToDocumentStage().with_(batch_size=1))
    p.add_stage(JsonlWriter(path=result_dir, write_kwargs={"force_ascii": False}))
    return p


print(build_pipeline(RESULT_DIR).describe())

2026-06-10 17:44:01.881 | INFO     | nemo_curator.pipeline.pipeline:add_stage:86 - Added stage 'CreateInitialManifestFleurs' to pipeline 'fleurs_tutorial'


2026-06-10 17:44:01.881 | INFO     | nemo_curator.pipeline.pipeline:add_stage:86 - Added stage 'ASR_inference' to pipeline 'fleurs_tutorial'


2026-06-10 17:44:01.882 | INFO     | nemo_curator.pipeline.pipeline:add_stage:86 - Added stage 'GetPairwiseWerStage' to pipeline 'fleurs_tutorial'


2026-06-10 17:44:01.882 | INFO     | nemo_curator.pipeline.pipeline:add_stage:86 - Added stage 'GetAudioDurationStage' to pipeline 'fleurs_tutorial'


2026-06-10 17:44:01.883 | INFO     | nemo_curator.pipeline.pipeline:add_stage:86 - Added stage 'PreserveByValueStage' to pipeline 'fleurs_tutorial'


2026-06-10 17:44:01.883 | INFO     | nemo_curator.pipeline.pipeline:add_stage:86 - Added stage 'AudioToDocumentStage' to pipeline 'fleurs_tutorial'


2026-06-10 17:44:01.884 | INFO     | nemo_curator.pipeline.pipeline:add_stage:86 - Added stage 'jsonl_writer' to pipeline 'fleurs_tutorial'


Pipeline: fleurs_tutorial
Description: Download FLEURS, run ASR, filter by WER
Stages: 7

Stage 1: CreateInitialManifestFleurs
  Resources: 1.0 CPUs
  Batch size: 4
  Outputs:
    Output columns: audio_filepath, text
Stage 2: ASR_inference
  Resources: 1.0 CPUs
    GPU Memory: 0.0 GB (1.0 GPUs)
  Batch size: 16
  Inputs:
    Required columns: audio_filepath
  Outputs:
    Output columns: audio_filepath, pred_text
Stage 3: GetPairwiseWerStage
  Resources: 1.0 CPUs
  Batch size: 1
  Inputs:
    Required columns: text, pred_text
  Outputs:
    Output columns: text, pred_text, wer_pct
Stage 4: GetAudioDurationStage
  Resources: 1.0 CPUs
  Batch size: 1
  Inputs:
    Required columns: audio_filepath
  Outputs:
    Output columns: duration
Stage 5: PreserveByValueStage
  Resources: 1.0 CPUs
  Batch size: 1
  Inputs:
    Required columns: wer_pct
  Outputs:
    Output columns: wer_pct
Stage 6: AudioToDocumentStage
  Resources: 1.0 CPUs
  Batch size: 1
Stage 7: jsonl_writer
  Resources: 1.0 CP

## Step 2: Execute the pipeline with both backends

`RayClient` manages the Ray cluster lifecycle (start/stop, port allocation, dashboard).
We run the pipeline with **both** backends and compare results and timing.

In [5]:
import time

from nemo_curator.backends.ray_data import RayDataExecutor

ray_client = RayClient()
ray_client.start()


def load_results(result_dir: str) -> list[dict]:
    """Read all JSONL files from a result directory."""
    data = []
    for fname in os.listdir(result_dir):
        if fname.endswith(".jsonl"):
            with open(os.path.join(result_dir, fname)) as f:
                data.extend(json.loads(line) for line in f if line.strip())
    return data


backends = {
    "xenna": XennaExecutor,
    "ray_data": RayDataExecutor,
}

run_results = {}

for name, executor_cls in backends.items():
    result_dir = os.path.join(RAW_DATA_DIR, f"result_{name}")
    pipeline = build_pipeline(result_dir)
    executor = executor_cls()

    t0 = time.time()
    pipeline.run(executor)
    elapsed = time.time() - t0

    data = load_results(result_dir)
    wers = [r.get("wer_pct", 0) for r in data]

    run_results[name] = {
        "time": elapsed,
        "samples": len(data),
        "mean_wer": sum(wers) / len(wers) if wers else 0,
        "total_dur": sum(r.get("duration", 0) for r in data),
        "data": data,
    }
    print(f"[{name}] {elapsed:.2f}s — {len(data)} samples, mean WER {run_results[name]['mean_wer']:.1f}%")

2026-06-10 17:44:02.005 | WARNING  | nemo_curator.core.client:start:121 - No monitoring services are running. Please run the `start_prometheus_grafana.py` script from nemo_curator/metrics folder to setup monitoring services separately.


2026-06-10 17:44:02.008 | INFO     | nemo_curator.core.utils:init_cluster:212 - Ray start command: ray start --head --node-ip-address 127.0.1.1 --port 6379 --metrics-export-port 8080 --dashboard-host 127.0.0.1 --dashboard-port 8265 --ray-client-server-port 10001 --temp-dir /tmp/ray --disable-usage-stats --block


2026-06-10 17:44:03,252	INFO usage_lib.py:448 -- Usage stats collection is disabled.
2026-06-10 17:44:03,252	INFO scripts.py:940 -- Local node IP: 127.0.1.1
2026-06-10 17:44:09,272	SUCC scripts.py:979 -- --------------------
2026-06-10 17:44:09,272	SUCC scripts.py:980 -- Ray runtime started.
2026-06-10 17:44:09,272	SUCC scripts.py:981 -- --------------------
2026-06-10 17:44:09,272	INFO scripts.py:983 -- Next steps
2026-06-10 17:44:09,272	INFO scripts.py:986 -- To add another node to this Ray cluster, run
2026-06-10 17:44:09,272	INFO scripts.py:989 --   ray start --address='127.0.1.1:6379'
2026-06-10 17:44:09,272	INFO scripts.py:1000 -- To connect to this Ray cluster:
2026-06-10 17:44:09,272	INFO scripts.py:1002 -- import ray
2026-06-10 17:44:09,272	INFO scripts.py:1003 -- ray.init(_node_ip_address='127.0.1.1')
2026-06-10 17:44:09,272	INFO scripts.py:1017 -- To submit a Ray job using the Ray Jobs CLI:
2026-06-10 17:44:09,272	INFO scripts.py:1018 --   RAY_API_SERVER_ADDRESS='http://127.

2026-06-10 17:44:10.337 | INFO     | nemo_curator.pipeline.pipeline:add_stage:86 - Added stage 'CreateInitialManifestFleurs' to pipeline 'fleurs_tutorial'


2026-06-10 17:44:10.339 | INFO     | nemo_curator.pipeline.pipeline:add_stage:86 - Added stage 'ASR_inference' to pipeline 'fleurs_tutorial'


...

Transcribing: 3it [00:00,  8.86it/s]
Transcribing: 4it [00:00,  8.77it/s]
(MapWorker(MapBatches(FleursAsrStageActor)) pid=2642215) [NeMo W 2026-06-10 17:45:33 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
(MapWorker(MapBatches(FleursAsrStageActor)) pid=2642215) [NeMo W 2026-06-10 17:45:33 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 0it [00:00, ?it/s]


Transcribing: 1it [00:00,  5.05it/s]
Transcribing: 2it [00:00,  6.62it/s]


Transcribing: 3it [00:00,  7.01it/s]
2026-06-10 17:45:33,600	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_8_0 execution finished in 25.80 seconds


2026-06-10 17:45:33.608 | INFO     | nemo_curator.backends.ray_data.executor:execute:98 - Pipeline completed. Final results: 50 tasks


[ray_data] 37.63s — 50 samples, mean WER 2.6%


In [6]:
MATCH_TOL = 0.1

print("\n" + "=" * 60)
print("Backend Comparison")
print("=" * 60)
print(f"{'':>12s}  {'Xenna':>10s}  {'Ray Data':>10s}  {'Match':>6s}")
print(f"{'Time (s)':>12s}  {run_results['xenna']['time']:10.2f}  {run_results['ray_data']['time']:10.2f}  {'':>6s}")
print(
    f"{'Samples':>12s}  {run_results['xenna']['samples']:10d}  {run_results['ray_data']['samples']:10d}"
    f"  {'✓' if run_results['xenna']['samples'] == run_results['ray_data']['samples'] else '✗':>6s}"
)
print(
    f"{'Mean WER':>12s}  {run_results['xenna']['mean_wer']:10.1f}  {run_results['ray_data']['mean_wer']:10.1f}"
    f"  {'✓' if abs(run_results['xenna']['mean_wer'] - run_results['ray_data']['mean_wer']) < MATCH_TOL else '✗':>6s}"
)
print(
    f"{'Audio (s)':>12s}  {run_results['xenna']['total_dur']:10.1f}  {run_results['ray_data']['total_dur']:10.1f}"
    f"  {'✓' if abs(run_results['xenna']['total_dur'] - run_results['ray_data']['total_dur']) < MATCH_TOL else '✗':>6s}"
)
speedup = run_results["ray_data"]["time"] / run_results["xenna"]["time"]
faster = "xenna" if speedup > 1 else "ray_data"
print(f"\n→ {faster} was {max(speedup, 1 / speedup):.1f}x faster on this dataset")

results = run_results["xenna"]["data"]


Backend Comparison
                   Xenna    Ray Data   Match
    Time (s)       45.89       37.63        
     Samples          50          50       ✓
    Mean WER         2.6         2.6       ✓
   Audio (s)       554.0       554.0       ✓

→ ray_data was 1.2x faster on this dataset


## Step 3: Load and inspect results

In [7]:
print(f"Total samples after filtering: {len(results)}")
print("\nSample entry:")
print(json.dumps(results[0], indent=2, ensure_ascii=False) if results else "No results")

Total samples after filtering: 50

Sample entry:
{
  "audio_filepath": "/home/aaftabv/grananary-v2/references/CuratorPRReviews/tutorials/audio/fleurs/example_audio/fleurs/hy_am/dev/14972895957050889721.wav",
  "text": "Երբ առաջին անգամ արտասահման գնացիք, մարդիկ հավանաբար համբերատար ու հասկացող էին՝ իմանալով, որ նոր երկրում ճանապարհորդները հարմարվելու կարիք ունեն:",
  "pred_text": "Երբ առաջին անգամ արտասահման գնացիք, մարդիկ հավանաբար համբերատար ու հասկացող էին՝ իմանալով, որ նոր երկրում ճանապարհորդները հարմարվելու կարիք ունեն։",
  "wer_pct": 5.2632,
  "duration": 12.0
}


## Step 4: Visualize results

### WER distribution

In [8]:
import matplotlib.pyplot as plt
import numpy as np

wers = [r.get("wer_pct", 0) for r in results]
durations = [r.get("duration", 0) for r in results]

if not wers:
    print("No results to visualize. Try relaxing WER_THRESHOLD or re-running the pipeline.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. WER histogram with threshold line
    ax = axes[0, 0]
    ax.hist(wers, bins=30, color="#4C72B0", edgecolor="white", alpha=0.85)
    ax.axvline(WER_THRESHOLD, color="#C44E52", linestyle="--", linewidth=2, label=f"Threshold ({WER_THRESHOLD}%)")
    ax.set_xlabel("WER (%)")
    ax.set_ylabel("Count")
    ax.set_title("WER Distribution")
    ax.legend()

    # 2. Duration distribution
    ax = axes[0, 1]
    ax.hist(durations, bins=30, color="#55A868", edgecolor="white", alpha=0.85)
    ax.set_xlabel("Duration (seconds)")
    ax.set_ylabel("Count")
    ax.set_title("Audio Duration Distribution")

    # 3. WER vs Duration scatter
    ax = axes[1, 0]
    scatter = ax.scatter(durations, wers, c=wers, cmap="RdYlGn_r", alpha=0.6, s=20, edgecolors="none")
    ax.axhline(WER_THRESHOLD, color="#C44E52", linestyle="--", linewidth=1.5, alpha=0.7)
    ax.set_xlabel("Duration (seconds)")
    ax.set_ylabel("WER (%)")
    ax.set_title("WER vs Duration")
    plt.colorbar(scatter, ax=ax, label="WER %")

    # 4. Pass rate at multiple thresholds
    ax = axes[1, 1]
    thresholds = [5, 10, 25, 50, 75, 100]
    pass_rates = [sum(1 for w in wers if w <= t) / len(wers) * 100 for t in thresholds]
    bars = ax.bar([str(t) for t in thresholds], pass_rates, color="#8172B2", edgecolor="white")
    ax.set_xlabel("WER Threshold (%)")
    ax.set_ylabel("Samples Passing (%)")
    ax.set_title("Dataset Yield by Threshold")
    for bar, rate in zip(bars, pass_rates, strict=True):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{rate:.0f}%", ha="center", fontsize=9)
    ax.set_ylim(0, 110)

    fig.suptitle(f"FLEURS {LANG} / {SPLIT} — {len(results)} samples (WER ≤ {WER_THRESHOLD}%)", fontsize=13, y=1.01)
    fig.tight_layout()
    plt.show()

    print(
        f"\nWER — min: {min(wers):.1f}%, max: {max(wers):.1f}%, mean: {np.mean(wers):.1f}%, median: {np.median(wers):.1f}%"
    )
    print(f"Duration — min: {min(durations):.2f}s, max: {max(durations):.2f}s, total: {sum(durations):.1f}s")

2026-06-10 17:45:33,961 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


2026-06-10 17:45:33,962 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


<Figure size 1400x1000 with 5 Axes>


WER — min: 0.0%, max: 5.3%, mean: 2.6%, median: 3.7%
Duration — min: 3.66s, max: 19.20s, total: 554.0s


## Step 5: Experiment with different thresholds

Try changing the WER threshold to see how it affects the dataset size:

In [9]:
thresholds = [5, 5.5, 10, 25, 50, 75]
for t in thresholds:
    passing = [r for r in results if r.get("wer_pct", 100) <= t]
    pct = len(passing) / len(results) * 100 if results else 0
    print(f"  WER ≤ {t:4.1f}%: {len(passing):4d} samples ({pct:.0f}%)")

  WER ≤  5.0%:   37 samples (74%)
  WER ≤  5.5%:   50 samples (100%)
  WER ≤ 10.0%:   50 samples (100%)
  WER ≤ 25.0%:   50 samples (100%)
  WER ≤ 50.0%:   50 samples (100%)
  WER ≤ 75.0%:   50 samples (100%)


## Cleanup

Shut down the Ray cluster started by `RayClient`.

In [10]:
ray_client.stop()

2026-06-10 17:45:37.588 | INFO     | nemo_curator.core.client:stop:205 - NeMo Curator has stopped the Ray cluster it started by killing the Ray GCS process. It is advised to wait for a few seconds before running any Ray commands to ensure Ray can cleanup other processes.If you are seeing any Ray commands like `ray status` failing, please ensure /tmp/ray/ray_current_cluster has correct information.
